In [8]:
!nvidia-smi

Sun Aug  2 10:16:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.86                 Driver Version: 581.86         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P8              3W /   75W |     647MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libs

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

import time

import shutil

import torchvision
import torchvision.utils
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torch.amp import autocast, GradScaler

import torchattacks
from torchattacks import PGD, FGSM
from torchsummary import summary
from sklearn.model_selection import train_test_split

# Data & Dataloader

In [2]:
torch.backends.cudnn.benchmark = True

batch_size = 64
num_epochs = 50

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

trainset = torchvision.datasets.ImageFolder(
    root="./data/GTSRB/Final_Training/Images",
    transform=train_transform
)

testset = torchvision.datasets.ImageFolder(
    root="./data/GTSRB/test",
    transform=test_transform
)

targets = np.array(trainset.targets)
class_sample_count = np.array([len(np.where(targets == t)[0]) for t in np.unique(targets)])
weight = 1. / class_sample_count
samples_weight = torch.from_numpy(weight[targets]).double()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

train_loader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_loader = torch.utils.data.DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# LNL Model

In [3]:
from LNL import LNL_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()

d:\HuuTuan\miniconda\envs\gpu2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\HuuTuan\miniconda\envs\gpu2\lib\site-packages\timm\models\helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
d:\HuuTuan\miniconda\envs\gpu2\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
d:\HuuTuan\miniconda\envs\gpu2\lib\site-packages\timm\models\registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.war

# Loss - Optimizer - Scheduler

In [4]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

scaler = GradScaler("cuda")

# Train

In [5]:
best_loss = float("inf")
checkpoint_dir = '../checkpoints'
checkpoint_path = os.path.join(checkpoint_dir, f"best_model.pth")

for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for i, (images, labels) in enumerate(train_loader):
        images = images.cuda(non_blocking=True)
        labels = labels.cuda(non_blocking=True)

        optimizer.zero_grad()
        with autocast(device_type="cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        if (i + 1) % 200 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}] "
                f"Iter [{i+1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}"
            )

    scheduler.step()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    lr = scheduler.get_last_lr()[0]

    print("-" * 60)
    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"LR: {lr:.7f}"
    )

    if train_loss < best_loss:
        best_loss = train_loss
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }, checkpoint_path)
        print("Best model saved.")

Epoch [1/50] Iter [200/613] Loss: 1.6015
Epoch [1/50] Iter [400/613] Loss: 0.7583
Epoch [1/50] Iter [600/613] Loss: 0.7258
------------------------------------------------------------
Epoch 01/50 | Loss: 1.4861 | Train Acc: 75.39% | LR: 0.0000999
Best model saved.
Epoch [2/50] Iter [200/613] Loss: 0.7177
Epoch [2/50] Iter [400/613] Loss: 0.7095
Epoch [2/50] Iter [600/613] Loss: 0.7032
------------------------------------------------------------
Epoch 02/50 | Loss: 0.7169 | Train Acc: 99.85% | LR: 0.0000996
Best model saved.
Epoch [3/50] Iter [200/613] Loss: 0.7078
Epoch [3/50] Iter [400/613] Loss: 0.7005
Epoch [3/50] Iter [600/613] Loss: 0.7177
------------------------------------------------------------
Epoch 03/50 | Loss: 0.7053 | Train Acc: 99.82% | LR: 0.0000991
Best model saved.
Epoch [4/50] Iter [200/613] Loss: 0.6968
Epoch [4/50] Iter [400/613] Loss: 0.6933
Epoch [4/50] Iter [600/613] Loss: 0.6910
------------------------------------------------------------
Epoch 04/50 | Loss: 0

# Test

### Test the model above

In [6]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.cuda()
        outputs = model(images)
        
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels.cuda()).sum()

print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

Standard accuracy: 99.41 %


### Test best checkpoint

In [7]:
from LNL import LNL_Ti as small

# Khởi tạo model
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()

# Load checkpoint
checkpoint = torch.load(checkpoint_path)

# Load trọng số
model.load_state_dict(checkpoint["model_state_dict"])

# Đưa lên GPU
model = model.cuda()

# Chế độ đánh giá
model.eval()

correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images = images.cuda()
        outputs = model(images)
        
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels.cuda()).sum()

print('Standard accuracy: %.2f %%' % (100 * float(correct) / total))

C:\Users\VIET THANG\AppData\Local\Temp\ipykernel_7996\1669174879.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Standard accuracy: 99.41 %


---